# PNS 2019 — saúde da pessoa idosa (60+)

Este notebook baixa os **microdados oficiais da Pesquisa Nacional de Saúde 2019 (IBGE)**, filtra os moradores com **60 anos ou mais** e gera um CSV pronto para análise.

- Fonte: https://ftp.ibge.gov.br/PNS/2019/Microdados/
- Resultado esperado: **43.554** moradores com 60+ anos, dos quais **22.728** responderam o questionário individual (morador selecionado).
- Tempo no Colab: cerca de 2 a 4 minutos (o arquivo do IBGE tem ~30 MB compactado e ~455 MB descompactado).

## 1. Baixar o script e gerar a base

In [ ]:
import os, urllib.request
if not os.path.exists("pns_idosos.py"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/adrianomanicoba/dados/main/pns2019_idosos/pns_idosos.py", "pns_idosos.py")

import pns_idosos
df, dic_var, dic_cat = pns_idosos.gerar("pns2019")

## 2. Conferência

In [ ]:
import pandas as pd
checks = {
    "moradores 60+ = 43.554": len(df) == 43_554,
    "moradores selecionados 60+ = 22.728": int(df.morador_selecionado.sum()) == 22_728,
    "população estimada 60+ ≈ 34,4 milhões": abs(df.V00281.sum() - 34_398_853) < 1_000,
}
for k, ok in checks.items():
    print(("✔ " if ok else "✘ ") + k)
print(f"População estimada com 60+ anos (peso V00281): {df.V00281.sum():,.0f}")

## 3. Como usar os pesos

A PNS tem amostra complexa. Use sempre um peso:

| Análise | Filtro | Peso |
|---|---|---|
| Variáveis respondidas por todos os moradores (módulos C a K, inclusive o **módulo K – saúde dos indivíduos com 60 anos ou mais**) | todos | `V00281` |
| Questionário individual (módulos L em diante: doenças crônicas, estilo de vida etc.) | `morador_selecionado == 1` | `V00291` |

Para erros-padrão e intervalos de confiança, use também o estrato `V0024` e a UPA `UPA_PNS` (ex.: pacote `samplics` em Python ou `survey` em R).

## 4. Dicionário: o que significa cada código

In [ ]:
def rotulo(v):
    if dic_var is None: return v
    s = dic_var.loc[dic_var.variavel == v, "descricao"]
    return s.iloc[0] if len(s) else v

def categorias(v):
    if dic_cat is None: return {}
    c = dic_cat[dic_cat.variavel == v]
    return dict(zip(c.valor.astype(str), c.descricao))

# variáveis do módulo K (saúde da pessoa idosa)
if dic_var is not None:
    display(dic_var[dic_var.variavel.str.match(r"^K\d")].reset_index(drop=True))

## 5. Exemplos de análise ponderada

In [ ]:
import numpy as np

def dist_ponderada(var, peso="V00281", base=df):
    b = base[[var, peso]].dropna()
    t = b.groupby(var)[peso].sum()
    out = (t / t.sum() * 100).round(1).rename("%").to_frame()
    cats = categorias(var)
    out.index = [cats.get(str(int(i)) if isinstance(i, float) and i.is_integer() else str(i), i) for i in out.index]
    print(f"{var} – {rotulo(var)}")
    return out

df["faixa_etaria"] = pd.cut(df.C008, [59, 69, 79, 200], labels=["60-69", "70-79", "80+"])
display((df.groupby("faixa_etaria", observed=True).V00281.sum() / df.V00281.sum() * 100).round(1).rename("% da população 60+"))
display(dist_ponderada("C006"))   # sexo
display(dist_ponderada("K001"))   # primeira variável do módulo K

## 6. Salvar o CSV (no seu computador ou no Google Drive)

In [ ]:
# Baixar para o computador:
from google.colab import files
files.download("pns2019/pns2019_idosos.csv.gz")
files.download("pns2019/dicionario_variaveis.csv")
files.download("pns2019/dicionario_categorias.csv")

# Ou copiar para o Google Drive (descomente):
# from google.colab import drive; drive.mount("/content/drive")
# !cp pns2019/*.csv* /content/drive/MyDrive/